# Module 4: Dense versus MoE on the GPU

In Module 3 you proved one request leaves the GPU memory-bound: to make one token, the server reads every weight once. This module gives that fact a picture and a name, then uses it to explain why a Mixture-of-Experts model generates faster than its size suggests. You plot the compute limit and the memory limit, find where they cross, and place decode far down the memory side. Then you derive, from the model configs, why an MoE moves fewer bytes per token than its total size. The plot runs live. The MoE comparison ships as commented-out code, because a 30B model does not fit the workshop card.

## Learning objectives
- Plot compute against memory bandwidth and read the crossover point for your card
- Place decode and prefill on that plot and say which is memory-bound
- Explain why decode reads every weight to make one token
- Derive active versus total parameters for a Mixture-of-Experts model and the bytes each moves per token
- Name the five ways to read fewer bytes, and which module covers each

- Section 5 measures the currently served 4B target model only. The 0.6B model is reserved for Module 6 as a speculative decoding draft model, not as a standalone served-model target.

References: [vLLM](https://docs.vllm.ai) &middot; [Anatomy of vLLM](https://blog.vllm.ai/2025/09/05/anatomy-of-vllm.html) &middot; [Qwen3-30B-A3B model card](https://huggingface.co/Qwen/Qwen3-30B-A3B) &middot; [RTX 4000 Ada datasheet](https://www.nvidia.com/en-us/products/workstations/rtx-4000/)

## Memory-bound design basics

Decode is memory-bound. To make one token, the server reads every weight in the model once, so the token rate is capped by how fast the card reads memory. Every technique in Part 2 is a way to read fewer bytes or reuse the bytes you read.

- Arithmetic intensity is the work done per byte read, in FLOPs per byte. You computed it in Module 3.
- Plot two limits against it: a sloped memory-bandwidth line and a flat compute line. Where they cross is the crossover point.
- Decode at batch 1 sits far down the memory side. Prefill sits near the compute line. Same card, two different places.

![The memory-bandwidth line and the compute line meeting at the crossover point, with decode at batch 1 far down the memory side and prefill near the top](images/04_dense_vs_moe_architecture.png)

## 1. Setup

The plot and the MoE math need only numpy and matplotlib. The dense measurement in section 5 uses your settings and client. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
# Imports, and your settings for the one live measurement later.
import os, sys, time
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from common.config import get_settings, build_client

settings = get_settings()
print("model   :", settings.model_name)
print("endpoint:", settings.vllm_host)

## 2. Memory-bound versus compute-bound, by hand (no GPU)

Plot the two limits. The compute line is flat at the card's peak FLOPs. The memory line slopes up with arithmetic intensity, at the card's bandwidth. Below the crossover you are memory-bound, above it compute-bound. Use the RTX 4000 Ada numbers: about 107 TFLOP/s dense FP16 and 360 GB/s.

In [ ]:
# The memory-bound vs compute-bound plot: where a job flips from waiting on memory to waiting on compute.
peak_tflops = 107.0     # RTX 4000 Ada, dense FP16. NOT 150: 427 on the sheet is FP8 with 2:1 sparsity.
bandwidth_tbs = 0.36    # 360 GB/s
crossover = peak_tflops / bandwidth_tbs
print(f"crossover point: {crossover:.0f} FLOPs/byte  (below this you are memory-bound)")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

intensity = np.logspace(-1, 3, 200)
attainable = np.minimum(peak_tflops, bandwidth_tbs * intensity)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.fill_between(intensity, 1e-2, attainable, where=intensity < crossover, color="#e74c3c", alpha=0.12)
ax.fill_between(intensity, 1e-2, attainable, where=intensity >= crossover, color="#2ecc71", alpha=0.12)
ax.plot(intensity, attainable, color="#1b1a3d", lw=2)
ax.axvline(crossover, ls="--", color="grey")
ax.scatter([1.0], [bandwidth_tbs], color="#c0392b", zorder=5, label="decode at batch 1 (memory-bound)")
ax.scatter([crossover * 3], [peak_tflops], color="#27ae60", zorder=5, label="prefill (compute-bound)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("arithmetic intensity (FLOPs/byte)")
ax.set_ylabel("attainable TFLOP/s")
ax.set_title(f"Memory-bound (red) below {crossover:.0f} FLOPs/byte, compute-bound (green) above")
ax.legend(); ax.grid(True, which="both", alpha=0.3); fig.tight_layout()
print(f"decode at intensity ~1 reaches {bandwidth_tbs:.2f} TFLOP/s, about {bandwidth_tbs / peak_tflops * 100:.2f}% of peak")

**What you should see:** a crossover near 297 FLOPs per byte, a red dot for decode sitting far down the sloped memory side, and a tiny fraction of peak compute reached. The tensor cores wait on memory. You measured this split in Module 3; this is the picture of it.

## 3. Why decode is memory-bound, in one sentence

To make one token, the server reads every weight in the model once, so your single-stream token rate is capped by how fast the card reads memory, not by how much math it can do. Every technique in Part 2 chips at that one limit.

## 4. MoE: the first way to read fewer weights (derive from the configs)

A dense model reads all its weights per token. A Mixture-of-Experts model holds many experts but routes each token to only a few, so it reads only the active ones. Total parameters set the memory footprint. Active parameters set the bandwidth cost, and bandwidth caps decode. Read the two from the configs and compare the bytes each moves per token.

In [ ]:
# Bytes moved per token at FP16. The MoE wins by reading only its active experts.
bpp = 2

dense4_read  = 4.0e9  * bpp     # dense 4B reads all 4B
dense30_read = 30.5e9 * bpp     # a dense 30B would read all 30.5B (hypothetical, for contrast)
moe_read     = 3.3e9  * bpp     # Qwen3-30B-A3B reads only ~3.3B active params (8 of 128 experts)
moe_store    = 30.5e9 * bpp     # but stores all 128 experts

print(f"dense 4B    : {dense4_read/1e9:5.1f} GB read per token")
print(f"dense 30B   : {dense30_read/1e9:5.1f} GB read per token  (hypothetical, for contrast)")
print(f"MoE 30B-A3B : {moe_read/1e9:5.1f} GB read per token, but {moe_store/1e9:.0f} GB stored")
print(f"=> the MoE reads {dense30_read/moe_read:.1f}x fewer bytes than a dense 30B, "
      f"so it decodes about that much faster")

**What you should see:** the MoE reads about 6.6 GB per token, roughly nine times less than a dense 30B, so it decodes about nine times faster while answering with 30B-scale quality. The catch is the footprint: all 128 experts sit in VRAM, about 61 GB at FP16, 30 GB at FP8, and 15 to 18 GB at INT4. It fits the 20 GB card only at INT4 and a small context, which is why the next section is a talk-through.

## 5. Compare dense-model sizes without switching the server

The workshop server should stay on the 4B target model. The 0.6B model is pre-cached for Module 6 as a speculative decoding draft model, so do not switch vLLM to serve it directly here.

Instead, measure the decode rate on the 4B model you are serving and compare it with the bandwidth estimate for a smaller dense model. The lesson is the same: a smaller dense model reads fewer weight bytes per token, so its memory-bound ceiling is higher. The Mixture-of-Experts model is the clever middle, 30B-scale quality at close to a small model's active-weight read, but it does not fit this card, so it stays a paper comparison here.

In [ ]:
# A small decode-rate probe: stream one answer and time the tokens.
from common.config import build_client
client = build_client(settings)

def decode_rate(model):
    start = time.time(); first = None; n = 0
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Write a detailed paragraph about GPU memory."}],
        max_tokens=200, temperature=0.0, stream=True,
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            if first is None:
                first = time.time()
            n += 1
    secs = (time.time() - first) if first else 0.0
    return n / secs if secs else 0.0

print(f"{settings.model_name}: about {decode_rate(settings.model_name):.0f} tokens/s")

**What you should see:** a decode rate near the bandwidth-over-weights ceiling from Module 2, the memory-bound floor for one request. The next cell estimates how a 0.6B dense model would move fewer bytes per token without changing the live server. That 0.6B model becomes useful later as a draft model for speculative decoding, where the 4B target still verifies the output.

In [ ]:
# Keep vLLM serving the 4B target model. The 0.6B model is a draft model for Module 6.
# Estimate the decode ceiling from memory bandwidth instead of switching models.
rtx4000_ada_bandwidth_gb_s = 360
bytes_per_param = 2  # BF16 baseline

served = settings.model_name
small_draft = "RedHatAI/Qwen3-0.6B-FP8-dynamic"
served_rate = decode_rate(served)

served_params_b = 4.0
small_active_params_b = 0.6
served_ceiling = rtx4000_ada_bandwidth_gb_s / (served_params_b * bytes_per_param)
small_ceiling = rtx4000_ada_bandwidth_gb_s / (small_active_params_b * 1)  # FP8 draft weights

print(f"{served}: measured about {served_rate:.0f} tokens/s")
print(f"{served}: bandwidth ceiling about {served_ceiling:.0f} tokens/s at BF16")
print(f"{small_draft}: draft-model ceiling about {small_ceiling:.0f} tokens/s at FP8")
print("Do not serve the draft model directly here; Module 6 uses it beside the 4B target.")

# The MoE would go here, but it does not fit a 20 GB card. On a 24 GB+ card at INT4:
#   serve Qwen/Qwen3-30B-A3B on a larger GPU and call decode_rate("Qwen/Qwen3-30B-A3B")
#   to compare its active-parameter read.

## 6. The reasoning tax

These are thinking models. By default they write a reasoning trace before the answer, which helps tool choice and hurts latency. At about 60 tokens per second, a 1,000-token trace is about 17 seconds on every agent step, before any tool call. The measurement cells in this workshop turn thinking off (`build_client` does it), so your numbers are the answer, not the thinking. Here is the cost when it is on.

In [ ]:
# Requires a live vLLM endpoint. Same question, thinking off (the measurement default) vs on.
import time
def timed(enable_thinking):
    t0 = time.time()
    r = client.chat.completions.create(
        model=settings.model_name,
        messages=[{"role": "user", "content": "Is a 4B or a 0.6B cheaper to run, and why? One short answer."}],
        max_tokens=500, temperature=0.0,
        extra_body={"chat_template_kwargs": {"enable_thinking": enable_thinking}},
    )
    return r.usage.completion_tokens, time.time() - t0

off_tok, off_s = timed(False)
on_tok, on_s = timed(True)
print(f"thinking OFF: {off_tok:>4} tokens in {off_s:.1f}s")
print(f"thinking ON : {on_tok:>4} tokens in {on_s:.1f}s  (+{on_tok - off_tok} reasoning tokens)")
print(f"at ~60 tok/s those extra tokens cost about {(on_tok - off_tok) / 60:.0f}s per agent step")

**What you should see:** thinking off returns a short answer fast. Thinking on emits a long trace first, often hundreds of tokens, so it takes several times longer. Multiply by every step of an agent loop and reasoning is a real latency budget. On a fixed context cap a runaway trace can even fill the output and return no answer. The lever: thinking off for steps that just need a tool call, on for steps that need a plan.

## 7. What this means for owning your inference

The memory-bound limit is how you pick a model for a card. A small dense model and a sparse MoE can hold the same latency on the same GPU for different reasons: the dense model is small enough to read fast, and the MoE is large but reads only its active experts. Knowing which you have tells you what to expect and what to tune.

## 8. Hand off to Omer

Decode is memory-bound. There are five ways to read fewer bytes or reuse the bytes you read, and you just saw the first.

- MoE: read only the active experts (this module).
- Quantization: fewer bytes per weight (Module 5, Omer).
- Attention kernels and fusion: fewer trips to GPU memory during attention (Module 7, Omer).
- Speculative decoding: more accepted tokens per target-model step when the drafter fits (Module 6, Omer).
- Continuous batching: one weight read shared across many requests, up to the knee (Module 7, Omer).

Each one points back to the plot you drew.

### The whole half on one card

Each layer you have met multiplies the others. This is the map to leave with.

| Layer | What it wins | Mechanism | For your agent |
|---|---|---|---|
| Algorithm | skip recompute | the KV cache (Module 3) | every decode step is cheap |
| Reuse | skip re-prefill | prefix caching (Module 3) | the resent system prompt is free |
| Architecture | a smaller cache | GQA, fewer KV heads (Module 3) | more agents fit per card |
| Precision | fewer bytes per weight | FP8 quantization (Module 5, Omer) | faster decode, more KV room |
| System | share the weight read | continuous batching (Module 7, Omer) | one card serves a fleet of agents |

Each layer multiplies the others. That is how one 20 GB card hosts a real agent.

## 9. Look forward: split prefill from decode

You proved prefill is compute-bound and decode is memory-bound. On one GPU they fight: a big prefill stalls everyone's decode. The next move in production serving is to stop sharing. Put prefill on one pool of GPUs and decode on another, and ship the KV cache between them over a fast link. That is disaggregated serving (vLLM's `--kv-transfer-config` with a NIXL connector, NVIDIA Dynamo, llm-d). You cannot run it on one 20 GB card, so this is the look-forward, not a lab. Be precise about the win: a single prefill-plus-decode pair does not raise throughput by itself. The gains come at fleet scale with smart routing, where you size and scale the two phases independently and stop a prefill spike from ever interrupting a decode.

![A prefill GPU builds the KV cache and ships it over NIXL to a decode GPU](images/04_disaggregation.png)


In [ ]:
# Not runnable on one GPU. The idea, and the flags for the day you have two.
print("On one GPU, a big prefill and an active decode share the same engine, so the prefill")
print("spike stalls the decode. Splitting them onto separate GPUs removes the interference.\n")
print("The flags, the day you have two GPUs:")
print("  prefill GPU:  --kv-transfer-config '{\"kv_connector\":\"NixlConnector\",\"kv_role\":\"kv_producer\"}'")
print("  decode  GPU:  --kv-transfer-config '{\"kv_connector\":\"NixlConnector\",\"kv_role\":\"kv_consumer\"}'")

## Things to know

- **The limit is per operation.** Prefill sits near the compute line, decode far down the memory side, on the same card.
- **The crossover point is a property of your GPU.** Put an H100's numbers in and it moves. The shape of the lesson does not.
- **MoE trades footprint for bandwidth.** You store every expert, and you read only the active ones. You pay in VRAM to save on the per-token read.
- **Use the dense figure for the card peak.** The RTX 4000 Ada is about 107 TFLOP/s dense FP16. The 427 on the spec sheet is FP8 with 2:1 sparsity.

## Try it yourself

**Recompute the crossover.** Put an H100's numbers in section 2 (about 990 TFLOP/s dense FP16, 3.35 TB/s) and read where the crossover moves. **Stretch:** mark where decode at batch 16 sits, and watch it climb the memory line toward the crossover.

**Compare another MoE.** Find a different MoE's total and active parameter counts in its config, and compute its bytes per token against a dense model of the same total size.

In [ ]:
# Change these, then run the cell.
your_peak_tflops = 990.0    # e.g. an H100, dense FP16
your_bw_tbs = 3.35          # e.g. an H100, 3.35 TB/s

your_crossover = your_peak_tflops / your_bw_tbs
print(f"crossover point: {your_crossover:.0f} FLOPs/byte")

## Summary

- Decode is memory-bound: one token reads every weight once, so memory bandwidth caps the rate.
- The plot puts compute against memory, and the crossover point is where they cross. Decode sits far down the memory side.
- A Mixture-of-Experts model reads only its active experts, so it moves fewer bytes per token than its total size and decodes faster.
- An MoE trades VRAM footprint for a smaller per-token read. It is the first of five ways to beat the memory limit.

## Next

**Hand off to Omer, Module 5: Quantization.** Decode is memory-bound, and quantization is the next way to read fewer bytes. Omer turns that into a decision framework: pick the right precision, understand the tradeoff, and measure what it costs in quality.